# Python Mastery — Module 7
# Unsupervised Learning and Dimensionality Reduction

This module teaches how to discover structure when the data has no known target label. It progresses from beginner concepts to practical and advanced considerations.

## Learning outcomes
- Explain supervised vs unsupervised learning.
- Prepare features for clustering and understand why scaling matters.
- Use K-means, hierarchical clustering, and DBSCAN.
- Choose and evaluate clustering approaches without relying on one score.
- Understand PCA, explained variance, and dimensionality reduction.
- Visualize high-dimensional data while avoiding overinterpretation.
- Detect unusual observations using isolation-based and density-based approaches.
- Build a practical workflow for segmentation and anomaly detection.

**Important:** Clusters are patterns found by an algorithm, not automatically real-world categories. Use domain knowledge and stability checks before assigning business meaning.

## 1. What is unsupervised learning?

In supervised learning, we learn from features and a known target. In unsupervised learning, we have features but no target labels supplied for training.

Common tasks:
- **Clustering:** group observations that are similar under a chosen representation and distance.
- **Dimensionality reduction:** represent many features with fewer variables while preserving selected structure.
- **Anomaly detection:** identify observations that look unusual relative to the data.
- **Association discovery:** identify items or events that co-occur.

The output depends strongly on feature engineering, scaling, distance, algorithm settings, and the data distribution. Different reasonable choices can produce different clusters.

## 2. Create a synthetic dataset

We will create data with several underlying groups. The known group labels are retained only for demonstration and comparison; clustering algorithms below will not use them during fitting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor, NearestNeighbors

RANDOM_STATE = 42
X_array, known_groups = make_blobs(
    n_samples=900, centers=4, n_features=5,
    cluster_std=[1.0, 1.3, 0.8, 1.1], random_state=RANDOM_STATE
)
feature_names = [f"feature_{i}" for i in range(X_array.shape[1])]
X = pd.DataFrame(X_array, columns=feature_names)
print("Shape:", X.shape)
X.head()

## 3. Why scaling matters

Distance-based algorithms can be dominated by a feature with a larger numeric scale. For example, income measured in dollars can dominate age measured in years even if age is just as useful.

- **StandardScaler:** centers each feature and scales it to unit standard deviation.
- **MinMaxScaler:** maps values to a chosen range, commonly 0 to 1.
- Robust scaling can be useful when extreme values distort mean and standard deviation.

Scaling is not always mandatory for every method, but it is usually important for K-means, hierarchical clustering, DBSCAN, KNN-like distances, and PCA. Fit the scaler on the appropriate development data and apply the same transformation consistently.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].boxplot(X.to_numpy())
axes[0].set_title("Before scaling")
axes[0].set_xlabel("Feature index")
axes[1].boxplot(X_scaled)
axes[1].set_title("After standard scaling")
axes[1].set_xlabel("Feature index")
plt.tight_layout()
plt.show()

## 4. K-means clustering

K-means partitions observations into `k` clusters by repeatedly:
1. assigning each point to its nearest centroid;
2. updating each centroid to the mean of its assigned points;
3. repeating until the assignments or centroids stabilize.

Important settings:
- `n_clusters`: number of clusters requested.
- `init`: centroid initialization strategy; `k-means++` is a common default.
- `n_init`: number of initializations; multiple starts reduce the chance of a poor local solution.
- `random_state`: reproducibility.

K-means works best for roughly compact, spherical groups under Euclidean distance. It can struggle with irregular shapes, very different cluster densities, and strong outliers.

In [ ]:
kmeans = KMeans(n_clusters=4, n_init=20, random_state=RANDOM_STATE)
kmeans_labels = kmeans.fit_predict(X_scaled)

print("Cluster sizes:")
print(pd.Series(kmeans_labels).value_counts().sort_index())
print("Inertia:", round(kmeans.inertia_, 2))

### Visualize the clusters in two dimensions

We use PCA below to project the data to two dimensions for visualization. The clustering itself was fitted on the full scaled feature set, not just the plotted two dimensions.

In [ ]:
pca_for_plot = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca_for_plot.fit_transform(X_scaled)

plt.figure(figsize=(7, 5))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=kmeans_labels, s=22, alpha=0.75)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means clusters shown in a 2D PCA projection")
plt.show()

## 5. Choosing k: elbow method and silhouette score

### Elbow method
Fit K-means for several values of `k` and plot inertia (within-cluster sum of squared distances). Inertia always decreases or stays the same as k increases, so look for a bend where improvements begin to slow.

### Silhouette score
For each observation, silhouette compares:
- how close it is to its own cluster;
- how far it is from the nearest alternative cluster.

Scores range from −1 to 1; higher values indicate better separation under the chosen distance. Silhouette favors certain cluster shapes and can be misleading for non-convex or density-based clusters. Do not treat either method as an automatic answer.

In [ ]:
ks = range(2, 10)
inertias, silhouettes = [], []
for k in ks:
    model = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_scaled)
    inertias.append(model.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(list(ks), inertias, marker="o")
axes[0].set_xlabel("k")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow method")
axes[1].plot(list(ks), silhouettes, marker="o")
axes[1].set_xlabel("k")
axes[1].set_ylabel("Silhouette score")
axes[1].set_title("Silhouette by k")
plt.tight_layout()
plt.show()

## 6. Evaluate clustering responsibly

Internal metrics can help compare candidate clusterings when no ground-truth labels exist:
- **Silhouette score:** higher is generally better.
- **Davies–Bouldin index:** lower is generally better.
- **Calinski–Harabasz index:** higher is generally better for many compact-cluster settings.

If labels exist only for retrospective evaluation, they can be compared after fitting using metrics such as Adjusted Rand Index (ARI). Do not use hidden labels to fit or tune the clustering if the goal is genuinely unsupervised discovery.

Also inspect:
- cluster sizes and profiles;
- stability under resampling and different seeds;
- usefulness to the business question;
- whether clusters are driven by an irrelevant variable or data quality artifact.

In [ ]:
metric_rows = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, n_init=15, random_state=RANDOM_STATE)
    labels = model.fit_predict(X_scaled)
    metric_rows.append({
        "k": k,
        "silhouette": silhouette_score(X_scaled, labels),
        "davies_bouldin": davies_bouldin_score(X_scaled, labels),
        "calinski_harabasz": calinski_harabasz_score(X_scaled, labels),
    })
pd.DataFrame(metric_rows).round(3)

## 7. Hierarchical clustering

Agglomerative clustering starts with each observation as its own cluster and repeatedly merges clusters based on a linkage rule.

Common linkage choices:
- **Ward:** merges to minimize the increase in within-cluster variance; commonly used with Euclidean distance.
- **Complete:** uses the maximum distance between points in two clusters.
- **Average:** uses average pairwise distance between clusters.
- **Single:** uses the minimum distance; can create chaining.

Advantages: no centroid initialization, and hierarchical structure can be explored at different cluster counts. Limitations: many implementations require substantial memory/time as the dataset grows, and results depend on distance and linkage.

In [ ]:
hierarchical = AgglomerativeClustering(n_clusters=4, linkage="ward")
hier_labels = hierarchical.fit_predict(X_scaled)
print("Cluster sizes:")
print(pd.Series(hier_labels).value_counts().sort_index())
print("Silhouette:", round(silhouette_score(X_scaled, hier_labels), 3))

## 8. DBSCAN: density-based clustering

DBSCAN groups points in dense regions and labels isolated points as noise (`-1`).

Key parameters:
- `eps`: neighborhood radius.
- `min_samples`: number of nearby observations needed to define a dense point.

Advantages:
- does not require specifying the number of clusters;
- can discover non-spherical shapes;
- identifies noise/outliers.

Limitations:
- results can change dramatically with `eps`;
- a single density scale may not fit clusters of very different densities;
- high-dimensional distances can become less informative.

Scale features and explore parameter sensitivity rather than assuming one setting is universally correct.

In [ ]:
moons, moons_true = make_moons(n_samples=700, noise=0.07, random_state=RANDOM_STATE)
moons_scaled = StandardScaler().fit_transform(moons)

kmeans_moons = KMeans(n_clusters=2, n_init=15, random_state=RANDOM_STATE).fit_predict(moons_scaled)
dbscan_moons = DBSCAN(eps=0.22, min_samples=6).fit_predict(moons_scaled)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].scatter(moons_scaled[:, 0], moons_scaled[:, 1], c=moons_true, s=12)
axes[0].set_title("Underlying groups (reference)")
axes[1].scatter(moons_scaled[:, 0], moons_scaled[:, 1], c=kmeans_moons, s=12)
axes[1].set_title("K-means")
axes[2].scatter(moons_scaled[:, 0], moons_scaled[:, 1], c=dbscan_moons, s=12)
axes[2].set_title("DBSCAN (-1 = noise)")
plt.tight_layout()
plt.show()
print("DBSCAN noise fraction:", round(np.mean(dbscan_moons == -1), 3))

### DBSCAN parameter sensitivity

A common practical aid is a k-distance plot: compute the distance to each point's kth nearest neighbor, sort those distances, and look for a possible bend. This is a heuristic, not a guaranteed way to select `eps`.

In [ ]:
min_samples = 6
neighbors = NearestNeighbors(n_neighbors=min_samples).fit(moons_scaled)
distances, _ = neighbors.kneighbors(moons_scaled)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(7, 4))
plt.plot(k_distances)
plt.xlabel("Points sorted by k-neighbor distance")
plt.ylabel(f"Distance to {min_samples}th nearest neighbor")
plt.title("DBSCAN k-distance diagnostic")
plt.show()

## 9. Compare clusterings with reference labels (when available)

For synthetic examples we know the labels used to generate the data. ARI compares two partitions while adjusting for chance; 1 means identical partitions and values near 0 indicate agreement similar to chance.

Use such metrics only when ground-truth labels are legitimately available for evaluation. In real unsupervised problems, labels may be absent, incomplete, subjective, or reflect a different goal.

In [ ]:
print("K-means ARI on synthetic blobs:", round(adjusted_rand_score(known_groups, kmeans_labels), 3))
print("Hierarchical ARI on synthetic blobs:", round(adjusted_rand_score(known_groups, hier_labels), 3))
print("K-means ARI on moons:", round(adjusted_rand_score(moons_true, kmeans_moons), 3))
print("DBSCAN ARI on moons (noise is a label):", round(adjusted_rand_score(moons_true, dbscan_moons), 3))

## 10. Principal Component Analysis (PCA)

PCA is a linear dimensionality-reduction technique. It finds orthogonal directions (principal components) that capture as much variance as possible:
- PC1 captures the greatest possible variance along one direction.
- PC2 captures the greatest remaining variance subject to being orthogonal to PC1.
- Later components capture progressively less variance.

PCA is unsupervised: it does not know which direction best predicts a target. A low-variance direction can still be predictive, so PCA is not guaranteed to improve supervised performance.

PCA is sensitive to feature scales; standardize features when their units differ and equalized scaling is appropriate.

In [ ]:
pca = PCA(random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
explained = pca.explained_variance_ratio_
cumulative = np.cumsum(explained)

pca_summary = pd.DataFrame({
    "component": [f"PC{i+1}" for i in range(len(explained))],
    "explained_variance_ratio": explained,
    "cumulative_explained_variance": cumulative,
})
pca_summary.round(3)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(np.arange(1, len(explained)+1), cumulative, marker="o")
plt.axhline(0.90, linestyle="--", label="90% variance")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.title("PCA explained variance")
plt.legend()
plt.show()

### PCA loadings and interpretation

A loading indicates how strongly an original feature contributes to a principal component. The sign of a component can flip without changing the underlying PCA solution; interpret relative directions and magnitudes, not the sign alone.

PCA components are combinations of original features and may be harder to explain to stakeholders. Fit PCA on training data and apply the same fitted transformation to validation/test data in predictive workflows.

In [ ]:
loadings = pd.DataFrame(
    pca.components_.T,
    index=feature_names,
    columns=[f"PC{i+1}" for i in range(len(feature_names))]
)
loadings.iloc[:, :3].round(3)

## 11. PCA as a preprocessing step for modeling

PCA can reduce feature dimensionality, noise, memory, or training time. However:
- PCA ignores target labels;
- components may be less interpretable;
- scaling and fitting must occur inside a pipeline to avoid leakage;
- performance must be compared with and without PCA using cross-validation.

The following example shows a safe pipeline structure. It is not a claim that PCA always improves classification.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.datasets import make_classification
from sklearn.pipeline import Pipeline

# Separate synthetic classification data for the supervised comparison.
Xc, yc = make_classification(
    n_samples=900, n_features=12, n_informative=6, n_redundant=2,
    weights=[0.85, 0.15], random_state=RANDOM_STATE
)
pca_models = {
    "Scale + logistic": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(max_iter=2500, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "Scale + PCA(90%) + logistic": Pipeline([
        ("scale", StandardScaler()),
        ("pca", PCA(n_components=0.90, random_state=RANDOM_STATE)),
        ("model", LogisticRegression(max_iter=2500, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = {}
for name, estimator in pca_models.items():
    scores = cross_val_score(estimator, Xc, yc, cv=cv, scoring="average_precision", n_jobs=-1)
    cv_scores[name] = (scores.mean(), scores.std())
pd.DataFrame(cv_scores, index=["mean average precision", "fold SD"]).T.round(3)

## 12. t-SNE: visualization, not a universal clustering solution

t-SNE is a nonlinear technique often used to visualize high-dimensional data in two dimensions. It aims to preserve local neighborhood relationships.

Cautions:
- distances between far-apart islands may not be meaningful;
- cluster sizes and gaps can be distorted;
- results depend on settings and random initialization;
- it is generally not a simple transform for new points in standard scikit-learn usage;
- do not infer that visible blobs prove real clusters exist.

Use t-SNE to explore local structure, not as the only evidence for segmentation.

In [ ]:
# Use a sample to keep t-SNE reasonably quick.
rng = np.random.default_rng(RANDOM_STATE)
sample_idx = rng.choice(len(X_scaled), size=500, replace=False)
tsne = TSNE(
    n_components=2, perplexity=30, init="pca",
    learning_rate="auto", random_state=RANDOM_STATE
)
X_tsne = tsne.fit_transform(X_scaled[sample_idx])

plt.figure(figsize=(7, 5))
plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=known_groups[sample_idx], s=18, alpha=0.75)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE visualization (colors show synthetic reference groups)")
plt.show()

## 13. Anomaly detection

An anomaly detector identifies observations that differ from a learned notion of normality. An anomaly is not automatically an error or fraud; it may be a rare but valid case.

### Isolation Forest
Uses randomized splits to isolate observations. Unusual points tend to be isolated with fewer splits.

### Local Outlier Factor (LOF)
Compares local density around a point with the density around its neighbors. It can find local anomalies, but behavior depends on neighborhood size and feature scaling.

### Practical cautions
- anomalies may be rare, so labels and evaluation can be difficult;
- scaling may matter depending on the method and features;
- choose contamination or threshold settings with care;
- investigate flagged records rather than automatically deleting them.

In [ ]:
iso = IsolationForest(
    n_estimators=200, contamination=0.03,
    random_state=RANDOM_STATE
)
iso_labels = iso.fit_predict(X_scaled)  # 1 = inlier, -1 = outlier
iso_scores = iso.decision_function(X_scaled)

lof = LocalOutlierFactor(n_neighbors=25, contamination=0.03)
lof_labels = lof.fit_predict(X_scaled)

print("Isolation Forest flagged:", int((iso_labels == -1).sum()))
print("LOF flagged:", int((lof_labels == -1).sum()))
print("Top 10 most unusual by Isolation Forest score (lower = more unusual):")
print(pd.Series(iso_scores).sort_values().head(10))

## 14. Clustering workflow for a real project

1. Define the business question and the unit of analysis.
2. Choose features that reflect the intended notion of similarity.
3. Remove identifiers that merely label rows, unless justified.
4. Investigate missingness, outliers, and data leakage.
5. Scale features when appropriate.
6. Establish several plausible candidate algorithms and settings.
7. Compare internal metrics, cluster sizes, stability, and domain usefulness.
8. Profile clusters using interpretable original features.
9. Test whether the segmentation is stable over time and useful for action.
10. Document limitations and avoid treating clusters as immutable identities.

For healthcare or other sensitive domains, check whether clusters simply reflect access differences, demographic proxies, or data collection artifacts.

## 15. Practice questions

Try to answer before reading the suggested solutions.

1. Why is scaling important for K-means and PCA?
2. What assumptions make K-means struggle with two-moons-shaped data?
3. What does DBSCAN label as `-1`?
4. What does a silhouette score measure at a high level?
5. Why should a high silhouette score not be the only basis for choosing a clustering?
6. What does PCA maximize?
7. Why can PCA remove predictive information?
8. Why should t-SNE plots not be interpreted as definitive proof of clusters?
9. What is the difference between an anomaly and an error?
10. Why should cluster stability be checked?

### Suggested answers

1. Features on larger numeric scales can dominate distance or variance calculations.
2. K-means represents clusters by centroids and favors compact, roughly spherical groups under Euclidean distance.
3. Points treated as noise that do not belong to a DBSCAN density-connected cluster.
4. It compares how similar a point is to its own cluster versus the nearest alternative cluster.
5. It favors some cluster geometries and may not reflect business usefulness, stability, or density-based structure.
6. PCA finds orthogonal directions that capture maximum variance in sequence.
7. PCA ignores the target; a low-variance direction can still contain useful predictive signal.
8. t-SNE distorts global geometry, and apparent islands or gaps can depend on settings.
9. An anomaly is unusual under a model or reference distribution; an error is invalid or incorrect data. An anomaly may be valid.
10. Unstable clusters may be artifacts of sampling, initialization, or arbitrary settings rather than robust patterns.

## 16. Capstone — member segmentation and anomaly review

### Scenario
A healthcare analytics team wants to group members by utilization and cost patterns, then flag unusually high-cost profiles for review.

### Suggested synthetic features
- visits in a period;
- emergency visits;
- inpatient admissions;
- medical cost;
- pharmacy cost;
- total cost;
- age band or product, handled thoughtfully.

### Tasks
1. Define the unit of analysis and feature window.
2. Inspect distributions, missing values, skew, and extreme values.
3. Scale the selected numeric features.
4. Compare K-means with hierarchical clustering and DBSCAN.
5. Use multiple diagnostics and inspect cluster sizes.
6. Profile clusters in the original feature units.
7. Run PCA for visualization and explain its limitations.
8. Flag anomalies with Isolation Forest and review the cases.
9. Write two business hypotheses, not unsupported claims.
10. Document risks, stability checks, and a plan for refreshing the segmentation.

Do not label clusters as “high risk,” “fraudulent,” or “healthy” without separate evidence and validation.

In [ ]:
# Build a synthetic member-level table for the capstone.
rng = np.random.default_rng(2027)
n_members = 1200
member_data = pd.DataFrame({
    "visits_12m": rng.poisson(4, n_members),
    "ed_visits_12m": rng.poisson(0.6, n_members),
    "ip_admissions_12m": rng.poisson(0.25, n_members),
    "medical_cost": rng.gamma(shape=2.0, scale=500, size=n_members),
    "pharmacy_cost": rng.gamma(shape=2.2, scale=160, size=n_members),
    "age": rng.integers(18, 91, n_members),
})
member_data["total_cost"] = member_data["medical_cost"] + member_data["pharmacy_cost"]
# Add a small number of deliberately extreme but not automatically invalid cases.
extreme_idx = rng.choice(member_data.index, size=12, replace=False)
member_data.loc[extreme_idx, "medical_cost"] *= 8
member_data.loc[extreme_idx, "total_cost"] = (
    member_data.loc[extreme_idx, "medical_cost"] +
    member_data.loc[extreme_idx, "pharmacy_cost"]
)
member_data.describe().round(2)

In [ ]:
capstone_features = [
    "visits_12m", "ed_visits_12m", "ip_admissions_12m",
    "medical_cost", "pharmacy_cost", "total_cost", "age"
]
capstone_scaler = StandardScaler()
member_scaled = capstone_scaler.fit_transform(member_data[capstone_features])

member_clusters = KMeans(n_clusters=4, n_init=20, random_state=RANDOM_STATE).fit_predict(member_scaled)
member_data["cluster"] = member_clusters

print("Cluster sizes:")
print(member_data["cluster"].value_counts().sort_index())
print("\nCluster profiles (original units):")
member_data.groupby("cluster")[capstone_features].median().round(1)

In [ ]:
capstone_iso = IsolationForest(contamination=0.02, random_state=RANDOM_STATE)
member_data["anomaly_flag"] = capstone_iso.fit_predict(member_scaled) == -1
print("Flagged cases:", int(member_data["anomaly_flag"].sum()))
member_data.loc[member_data["anomaly_flag"], capstone_features].sort_values("total_cost", ascending=False).head(10)

## 17. Readiness checklist

- [ ] Explain supervised versus unsupervised learning.
- [ ] Explain why scaling matters for distance-based methods and PCA.
- [ ] Fit K-means and use elbow/silhouette diagnostics.
- [ ] Explain hierarchical clustering and linkage choices.
- [ ] Tune DBSCAN's `eps` and `min_samples` thoughtfully.
- [ ] Interpret PCA components and cumulative explained variance.
- [ ] Explain limitations of t-SNE visualizations.
- [ ] Compare clustering metrics with stability and business usefulness.
- [ ] Explain Isolation Forest and LOF at a high level.
- [ ] Distinguish an anomaly from invalid data.
- [ ] Profile clusters in original units and document uncertainty.

### Next module
**Module 8 — Deep Learning Foundations:** neural networks, tensors, forward propagation, loss functions, gradient descent, backpropagation, optimizers, regularization, and training neural networks with practical examples.